# HTTP, REST, and Status Codes

This notebook covers:

1. Identify the HTTP methods (`GET`, `POST`, `PUT`, `PATCH`, `DELETE`) and their semantics
2. Read and write status codes correctly (2xx / 3xx / 4xx / 5xx)
3. Recognize the REST constraints (statelessness, uniform interface, resources)
4. Tell idempotent methods from non-idempotent ones — and why it matters

**Scope**: Conceptual + `httpx` calls against `httpbin.org`. No FastAPI yet — that starts in notebook 02.

**Requires internet**: this is the only notebook in the series that calls a public endpoint. Every other notebook runs in-process with `TestClient`.

In [ ]:
import httpx

# httpbin.org echoes back whatever you send — perfect for inspecting HTTP behavior
# without standing up our own server.
BASE = "https://httpbin.org"

# Short timeout so cells fail fast if you're offline rather than hanging the kernel.
client = httpx.Client(base_url=BASE, timeout=10.0)

# If you see SSL: CERTIFICATE_VERIFY_FAILED here, your network likely has a TLS
# interception proxy. Quick fix in a controlled environment: BASE = "http://httpbin.org".
# Proper fix: install your corporate CA into the certifi bundle (Google your org's name).

## 1. The HTTP Request / Response Cycle

An HTTP exchange is a *request* from a client and a *response* from a server. Every request is a self-contained envelope: a method, a target URL, headers, and (sometimes) a body. The response has the same shape: a status line, headers, and (usually) a body.

There is no implicit continuity between requests — the protocol is **stateless**. Anything the server needs to remember about you must travel in the request itself (cookies, auth headers, query params). That single property is what lets REST APIs scale horizontally: any replica can serve any request.

In [ ]:
resp = client.get("/get", params={"hello": "world"})

print("status :", resp.status_code, resp.reason_phrase)
print("\nheaders:")
for k, v in resp.headers.items():
    print(f"  {k}: {v}")

print("\nbody (echoed by httpbin):")
print(resp.json())

## 2. Methods and Their Semantics

| Method  | Purpose            | Safe* | Idempotent** |
|---------|--------------------|:-----:|:------------:|
| GET     | Read a resource    | yes   | yes          |
| HEAD    | Read headers only  | yes   | yes          |
| POST    | Create / submit    | no    | no           |
| PUT     | Replace a resource | no    | yes          |
| PATCH   | Partial update     | no    | sometimes    |
| DELETE  | Remove a resource  | no    | yes          |

\*Safe = does not change server state. \**Idempotent = repeating the call has the same effect as making it once.

`GET` is safe and idempotent. `POST` is neither — two POSTs to `/orders` create two orders. `PUT` and `DELETE` are idempotent: replacing a record with the same value twice, or deleting it twice, leaves the same end state.

These properties drive real decisions: retries, caching, and form-submission behavior all depend on them.

In [ ]:
# Round-trip each method against httpbin to see what comes back.
# httpbin exposes a path per method (/get, /post, /put, ...).
for method in ["GET", "POST", "PUT", "PATCH", "DELETE"]:
    path = "/" + method.lower()
    body = None if method == "GET" else {"sample": 1}
    r = client.request(method, path, json=body)
    print(f"{method:7} {path:9} -> {r.status_code} {r.reason_phrase}")

## 3. Status Codes: 2xx, 3xx, 4xx, 5xx

| Family | Meaning        | Examples |
|--------|----------------|----------|
| 2xx    | Success        | 200 OK, 201 Created, 204 No Content |
| 3xx    | Redirection    | 301 Moved Permanently, 304 Not Modified |
| 4xx    | Client error   | 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 422 Unprocessable Entity |
| 5xx    | Server error   | 500 Internal Server Error, 502 Bad Gateway, 503 Service Unavailable |

A good API picks the *most specific* code available:

- `404` — "resource doesn't exist"
- `403` — "exists, but you can't have it"
- `401` — "I don't know who you are"
- `422` — "I understood the request shape but the values failed validation"

Mixing these is a real source of debugging pain. FastAPI gives us `422` for free on Pydantic validation failures (notebook 1.3) — keep that distinction.

In [ ]:
# httpbin returns whatever status code you ask for — useful for testing client behavior.
# follow_redirects=False so 3xx responses don't get auto-followed and hide the original code.
for code_ in [200, 201, 204, 301, 400, 401, 403, 404, 418, 422, 500, 503]:
    r = client.get(f"/status/{code_}", follow_redirects=False)
    print(f"requested {code_} -> got {r.status_code} {r.reason_phrase}")

## 4. Idempotency

A method is **idempotent** when making the call N times has the same observable effect on server state as making it once.

- `GET /assets/AAPL` — read, no state change → idempotent.
- `PUT /assets/AAPL` with `{"price": 200}` — sets price to 200; repeating still sets it to 200 → idempotent.
- `POST /assets` with `{...}` — creates a new asset each time → **not** idempotent.
- `DELETE /assets/AAPL` — gone after the first call; the second call returns 404 but the *state* is the same → idempotent.

Why this matters: **network retries**. A flaky connection can cause a client (or a proxy, or a load balancer) to silently retry a request. If the request is idempotent, retrying is safe. If it isn't, you may charge a customer twice. Production clients (and the HTTP spec) only auto-retry idempotent methods.

In [ ]:
# Two POSTs are two independent events. httpbin has no real state, but conceptually
# each one would create a new row server-side.
post1 = client.post("/post", json={"order_id": 1}).json()
post2 = client.post("/post", json={"order_id": 1}).json()
print("two POSTs — server would treat each as a separate creation:")
print("  post1 body:", post1["json"])
print("  post2 body:", post2["json"])

# Two PUTs converge on the same end state. Whichever order they arrive in, the
# server's record ends up identical — safe to retry.
put1 = client.put("/put", json={"price": 200}).json()
put2 = client.put("/put", json={"price": 200}).json()
print("\ntwo PUTs — server's end state is the same either way:")
print("  put1 body:", put1["json"])
print("  put2 body:", put2["json"])

## 5. REST as a Set of Constraints

Roy Fielding's dissertation (2000) defined REST as six architectural *constraints*, not as "HTTP + JSON." The ones that matter for day-to-day API design:

- **Statelessness** — every request carries everything the server needs. No session affinity, no sticky load balancers required.
- **Uniform interface** — resources are identified by URLs, manipulated via a small fixed set of methods, and exchanged as self-descriptive representations.
- **Cacheability** — responses declare whether and how they can be cached (`Cache-Control`, `ETag`, `Last-Modified`).
- **Layered system** — clients can't tell whether they're talking to the origin or a proxy / gateway / CDN.
- **Client-server separation** — UI and storage evolve independently.
- **Code-on-demand** — optional, rarely used in practice.

Most "REST APIs" in the wild satisfy two or three of these and call it a day. That's fine — but knowing which constraint you're violating tells you what scaling problem you'll hit. A stateful API can't be load-balanced naively; an uncacheable one will hammer your database.

## 6. URLs, Resources, Representations

A **resource** is an abstract thing (an asset, a portfolio). A **URL** is its identifier. A **representation** is one concrete serialization of it (JSON, CSV, XML, ...).

The same resource can have many representations. Clients ask for the one they want via the `Accept` header — this is **content negotiation**. Servers signal what they actually returned with `Content-Type`.

In our portfolio domain: `GET /assets/AAPL` might return JSON by default, but `Accept: text/csv` could return the same resource as a CSV row. We'll do exactly this kind of dual-representation endpoint in notebook 2.2.

In [ ]:
# Ask httpbin to advertise a different Content-Type, simulating a server that
# offers multiple representations of the same resource.
for accept in ["application/json", "text/html", "application/xml"]:
    r = client.get("/response-headers", params={"Content-Type": accept})
    print(f"Accept={accept!r:22} -> server says Content-Type: {r.headers['content-type']}")

## Key Takeaways

- **HTTP is stateless** — every request stands alone. No hidden conversation.
- **Methods carry semantics** — pick the right one (`GET` to read, `POST` to create, `PUT` to replace, `PATCH` to update, `DELETE` to remove). The wrong method silently breaks caches, retries, and CDNs.
- **Status codes are the API's primary signal** — use the most specific code. Never return `200 OK` with `{"error": ...}` in the body.
- **Idempotency is a contract with the network**, not just a definition. It's what makes safe retries possible.
- **REST is a set of constraints**, not a checklist. Statelessness and the uniform interface are the load-bearing ones; cacheability is the one most often skipped.

Next up: notebook 1.2 puts a real FastAPI app behind these rules.

## Exercises

**1. Classify each operation.** Pick the right method, and label it safe / idempotent / neither.

- "Return all assets in a portfolio."
- "Add a transaction to a portfolio."
- "Set a user's email address to `x@y.com`."
- "Apply a one-time 5% discount to an order."

**2. Pick the right status code** for each:

- The client sent a body with a missing required field.
- The client is authenticated but lacks permission for this resource.
- The endpoint exists but is temporarily under maintenance.
- The client successfully created a new asset.
- The client asked for `/assets/ZZZZ` but no such asset exists.

**3. Cache reasoning.** Which of these responses should be cacheable, and for how long? Why?

- `GET /assets/AAPL/price` — current live ticker price.
- `GET /assets/AAPL/metadata` — name, sector, listing exchange.
- `GET /portfolios/me` — the current user's portfolio.

(Answers aren't included — the goal is to argue them out. Check your reasoning against the cacheability constraint in §5.)

In [ ]:
client.close()